In [ ]:
# %pip install -q -U langchain langchain-openai langchain-community langgraph ddgs python-dotenv
# %pip install yfinance


In [ ]:
import os
import json
from typing import TypedDict, List, Literal

import yfinance as yf
import re

# LangChain & LangGraph 관련 라이브러리
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchResults
from langgraph.graph import StateGraph, START, END

In [ ]:
OPENAI_API_KEY = ""
LANGSMITH_API_KEY = ""

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"]    = LANGSMITH_API_KEY
os.environ.setdefault("LANGCHAIN_PROJECT", "ss_genai")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0.5)

llm

In [ ]:
class DebateTurn(TypedDict):
    """개별 발언을 구조화하기 위한 서브 딕셔너리"""
    speaker: Literal["bull", "bear"]
    turn: int
    content: str

class AgentState(TypedDict):
    topic: str
    collected_docs: List[str]
    debate_history: List[DebateTurn]  # 문자열 리스트가 아닌 구조화된 리스트로 변경!
    turn_count: int
    max_turns: int
    final_summary: str 
    rejection_feedback: str  # [추가] 반려 사유를 담는 변수

In [ ]:
def initialize(state: AgentState):
    """상태 초기화 및 주제/턴 수 설정"""
    print("\n[초기화] 에이전트 상태를 세팅합니다...")
    # 기본 2턴 동안 공방을 주고받도록 설정
    return {"turn_count": 0, "max_turns": 3, "debate_history": [], "collected_docs": []}

In [ ]:
def research_node(state: AgentState):
    """yfinance(재무/뉴스)와 DuckDuckGo(동적 검색)를 결합한 하이브리드 리서치 노드"""
    print("\n[리서치 에이전트] 시장 데이터 및 뉴스를 수집 중입니다...")
    
    topic = state.get("topic", "")
    history = state.get("debate_history", [])
    collected_docs = state.get("collected_docs", [])
    current_turn = state.get("turn_count", 0)
    
    formatted_result = ""
    
    # ---------------------------------------------------------
    # 턴 0: yfinance를 활용한 펀더멘털 및 핵심 뉴스 딥다이브
    # ---------------------------------------------------------
    print("  -> [Turn 0] yfinance API로 재무제표 및 공식 뉴스 수집 중...")
    
    # 1. 괄호 안의 티커(Ticker) 추출 (예: "NVIDIA (NVDA)" -> "NVDA")
    match = re.search(r'\((.*?)\)', topic)
    ticker_symbol = match.group(1) if match else topic.split()[0]
    
    try:
        ticker = yf.Ticker(ticker_symbol)
        info = ticker.info
        news = ticker.news
        
        # 2. 핵심 재무 지표 추출
        financial_data = (
            f"[기업 펀더멘털 요약: {ticker_symbol}]\n"
            f"- 현재가: {info.get('currentPrice', 'N/A')} {info.get('financialCurrency', 'USD')}\n"
            f"- 52주 최고/최저: {info.get('fiftyTwoWeekHigh', 'N/A')} / {info.get('fiftyTwoWeekLow', 'N/A')}\n"
            f"- PER (Trailing): {info.get('trailingPE', 'N/A')}\n"
            f"- PBR (PriceToBook): {info.get('priceToBook', 'N/A')}\n"
            f"- 영업이익률(Operating Margin): {info.get('operatingMargins', 0) * 100:.2f}%\n"
            f"- 매출 성장률(Revenue Growth): {info.get('revenueGrowth', 0) * 100:.2f}%\n"
        )
        
        # 3. 최신 뉴스 추출 (최대 3개)
        news_data = "[최신 공식 금융 뉴스]\n"
        for i, n in enumerate(news[:3]):
            news_data += f"{i+1}. 제목: {n.get('title', '제목 없음')}\n   출판사: {n.get('publisher', '알 수 없음')}\n   링크: {n.get('link', '')}\n"
        
        formatted_result = f"--- [턴 1 초기 수집 데이터 (yfinance)] ---\n{financial_data}\n{news_data}"
        
    except Exception as e:
        print(f"  -> yfinance 호출 실패: {e}")
        formatted_result = "--- [턴 1 초기 수집 데이터] ---\n재무 데이터 수집에 실패했습니다. 기존 지식을 활용하세요."

    
    # 4. 결과 저장
    return {"collected_docs": collected_docs + [formatted_result]}

In [ ]:
def bull_agent(state: AgentState):
    """낙관적/가치 상승 관점 분석 (Bull)"""
    print("\n[Bull 에이전트] 낙관적 투자 의견을 생성 중입니다...")
    topic = state.get("topic", "")
    collected_docs = state.get("collected_docs", [])
    history = state.get("debate_history", [])
    current_turn = state.get("turn_count", 0) + 1 
    
    history_str = "\n".join([f"{item['speaker'].upper()} (턴 {item['turn']}): {item['content']}" for item in history])
    if not history_str: history_str = "토론의 첫 번째 발언입니다."
    docs_str = "\n".join(collected_docs) if collected_docs else "데이터 없음"

    llm = ChatOpenAI(model="gpt-4o", temperature=0.7)
    system_prompt = """당신은 월스트리트 최고의 '낙관론자(Bull)' 수석 애널리스트입니다.
    [행동 원칙]
    1. 수집된 데이터를 바탕으로 기업의 내재 가치, 성장성, 긍정적 모멘텀에 집중하세요.
    2. 이전 토론 내역에서 상대방(Bear)의 비관적 논리가 있다면, 데이터를 근거로 반박하세요.
    
    [수집된 데이터]
    {collected_docs}
    
    [이전 토론 내역]
    {debate_history}
    """
    
    # 반려를 당해서 다시 돌아온 경우, 시스템 프롬프트에 경고장 추가!
    rejection_feedback = state.get("rejection_feedback", "")
    if rejection_feedback:
        print("  -> ⚠️ Bull Agent가 반려 사유를 반영하여 주장을 수정합니다.")
        system_prompt += f"""
        \n[🚨 팩트체크 부서의 반려 경고 🚨]
        직전 당신의 발언이 다음과 같은 사유로 반려되었습니다:
        {rejection_feedback}
        위 지적 사항을 겸허히 수용하고, 오류를 수정한 새로운 논리를 제시하세요.
        """

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "위 정보를 바탕으로 {topic}에 대한 긍정적 논리를 전개하고 상대방을 반박하세요.")
    ])
    
    response = (prompt | llm).invoke({"topic": topic, "collected_docs": docs_str, "debate_history": history_str})
    
    new_turn: DebateTurn = {"speaker": "bull", "turn": current_turn, "content": response.content}
    return {"debate_history": history + [new_turn]}

In [ ]:
def bear_agent(state: AgentState):
    """비관적/리스크 관점 분석 (Bear)"""
    print("\n[Bear 에이전트] 리스크 및 비관적 의견을 생성 중입니다...")
    topic = state.get("topic", "")
    collected_docs = state.get("collected_docs", [])
    history = state.get("debate_history", [])
    current_turn = state.get("turn_count", 0) + 1 
    
    history_str = "\n".join([f"{item['speaker'].upper()} (턴 {item['turn']}): {item['content']}" for item in history])
    docs_str = "\n".join(collected_docs) if collected_docs else "데이터 없음"

    llm = ChatOpenAI(model="gpt-4o", temperature=0.7)
    system_prompt = """당신은 철저한 '비관론자 및 리스크 관리자(Bear)' 수석 애널리스트입니다.
    [행동 원칙]
    1. 수집된 데이터를 바탕으로 거시 경제 리스크, 재무적 취약점, 경쟁 심화 요인에 집중하세요.
    2. 이전 토론 내역에서 상대방(Bull)의 과도한 낙관론을 데이터를 근거로 날카롭게 반박하세요.
    
    [수집된 데이터]
    {collected_docs}
    
    [이전 토론 내역]
    {debate_history}
    """
    
    # 반려를 당해서 다시 돌아온 경우, 시스템 프롬프트에 경고장 추가!
    rejection_feedback = state.get("rejection_feedback", "")
    if rejection_feedback:
        print("  -> ⚠️ Bull Agent가 반려 사유를 반영하여 주장을 수정합니다.")
        system_prompt += f"""
        \n[🚨 팩트체크 부서의 반려 경고 🚨]
        직전 당신의 발언이 다음과 같은 사유로 반려되었습니다:
        {rejection_feedback}
        위 지적 사항을 겸허히 수용하고, 오류를 수정한 새로운 논리를 제시하세요.
        """

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "위 정보를 바탕으로 {topic}에 대한 리스크를 강조하고 상대방의 낙관론을 반박하세요.")
    ])
    
    response = (prompt | llm).invoke({"topic": topic, "collected_docs": docs_str, "debate_history": history_str})
    
    new_turn: DebateTurn = {"speaker": "bear", "turn": current_turn, "content": response.content}
    return {"debate_history": history + [new_turn], "turn_count": current_turn}

In [ ]:
def verify_node(state: AgentState):
    print("\n[Verify Node] 직전 턴의 발언을 팩트체크합니다...")
    topic = state.get("topic", "")
    history = state.get("debate_history", [])
    current_turn = state.get("turn_count", 0)
    
    # 직전 턴의 Bull과 Bear 발언 가져오기 (가장 최근 2개)
    recent_speeches = history[-2:] if len(history) >= 2 else history
    claims = "\n".join([f"{item['speaker']}: {item['content']}" for item in recent_speeches])
    
    llm = ChatOpenAI(model="gpt-4o", temperature=0.0) # 판정은 냉정해야 하므로 온도 0
    search_tool = DuckDuckGoSearchResults(max_results=2)
    
    # 1. 팩트체크용 데이터 검색
    query_prompt = ChatPromptTemplate.from_messages([
        ("system", "다음 발언들의 진위를 확인하기 위한 영문 검색어 1개만 작성하세요. 따옴표 없이 작성하세요."),
        ("human", claims)
    ])
    search_query = (query_prompt | llm).invoke({}).content.strip()
    search_results = search_tool.invoke(search_query)
    
    # 2. PASS / FAIL 판정 (가장 중요한 부분!)
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 엄격하고 냉정한 팩트체커입니다. 
        [수집된 팩트]와 [에이전트의 발언]을 비교하세요.
        - 발언에 명백한 거짓, 없는 수치 지어내기(Hallucination), 심각한 논리적 오류가 있다면 'FAIL: (구체적인 반려 사유)' 형태로 출력하세요.
        - 수용 가능한 수준의 사실 기반 의견이라면 'PASS'라고만 출력하세요."""),
        ("human", f"[수집된 팩트]\n{search_results}\n\n[에이전트의 발언]\n{claims}")
    ])
    
    eval_result = (eval_prompt | llm).invoke({}).content.strip()
    
    # 3. 판정 결과에 따른 상태 업데이트 (Rollback 로직)
    if eval_result.startswith("FAIL"):
        print(f"  🚨 [판정: 반려] {eval_result}")
        # 오류가 있는 최근 2개의 발언을 역사에서 지워버립니다 (Rollback)
        rollback_history = history[:-2]
        
        return {
            "debate_history": rollback_history, 
            "rejection_feedback": eval_result # 반려 사유를 다음 에이전트에게 전달
            # 주의: turn_count는 증가시키지 않습니다! (다시 써야 하므로)
        }
    else:
        print("  ✅ [판정: 통과] 팩트 검증 완료")
        # 통과했을 때만 턴 카운트를 1 증가시킵니다.
        return {
            "rejection_feedback": "", 
            "verified_facts": state.get("verified_facts", []) + [search_results],
            "turn_count": current_turn + 1 
        }

In [ ]:
def save_agent_results(state: AgentState):
    """결과 데이터를 사람이 읽기 편한 마크다운(.md) 형식으로 저장하는 함수"""
    topic_clean = state['topic'].replace(" ", "_").replace("/", "_")
    folder_name = f"result_{topic_clean}"
    
    # 결과 저장용 폴더 생성
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    
    # ---------------------------------------------------------
    # (1) 수집된 데이터 저장 (깔끔한 구분선 추가)
    # ---------------------------------------------------------
    with open(f"{folder_name}/1_collected_docs.md", "w", encoding="utf-8") as f:
        f.write(f"# 🔍 수집된 리서치 데이터: {state['topic']}\n\n")
        for doc in state['collected_docs']:
            f.write(f"{doc}\n\n---\n\n") # 문서마다 구분선(---)을 추가하여 가독성 확보
    
    # ---------------------------------------------------------
    # (2) 토론 내역 저장 (JSON -> 채팅 대본 형식으로 변환)
    # ---------------------------------------------------------
    with open(f"{folder_name}/2_debate_history.md", "w", encoding="utf-8") as f:
        f.write(f"# 💬 에이전트 토론 기록: {state['topic']}\n\n")
        
        for turn in state['debate_history']:
            # 화자에 따라 이모지와 이름 다르게 설정
            if turn['speaker'] == 'bull':
                speaker_name = "🐂 Bull (낙관론자)"
            else:
                speaker_name = "🐻 Bear (비관론자)"
                
            f.write(f"### {speaker_name} - 턴 {turn['turn']}\n")
            f.write(f"{turn['content']}\n\n")
    
    # ---------------------------------------------------------
    # (3) 최종 리포트 저장 (기존 유지)
    # ---------------------------------------------------------
    with open(f"{folder_name}/3_final_report.md", "w", encoding="utf-8") as f:
        f.write(f"# 📄 최종 투자 분석 리포트: {state['topic']}\n\n")
        f.write(state['final_summary'])
        
    print(f"\n✅ 가독성이 개선된 파일들이 '{folder_name}' 폴더에 저장되었습니다.")

In [ ]:
def summary_agent(state: AgentState):
    """최종 투자 의견 요약 리포트 생성"""
    print("\n[요약 에이전트] 최종 투자 의견 리포트를 작성 중입니다...")
    history = state.get("debate_history", [])
    topic = state.get("topic", "")
    
    # 딕셔너리 구조를 활용하여 진영별 의견 완벽 분리 추출
    bull_opinions = "\n".join([f"- {turn['content']}" for turn in history if turn["speaker"] == "bull"])
    bear_opinions = "\n".join([f"- {turn['content']}" for turn in history if turn["speaker"] == "bear"])
    
    llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 중립적이고 객관적인 최종 투자 의사결정권자(CIO)입니다."),
        ("human", """다음은 {topic}에 대한 치열한 토론 결과입니다.
        
        [긍정적 요인 (Bull)]
        {bull_opinions}
        
        [부정적 요인 (Bear)]
        {bear_opinions}
        
        위 내용을 종합하여, {topic}에 대해 투자를 진행해야 할지 말아야 할지 결론을 내리고, 그 이유를 3~4문단의 깔끔한 텍스트로 요약해 주세요.
        """)
    ])
    
    final_report = (prompt | llm).invoke({"topic": topic, "bull_opinions": bull_opinions, "bear_opinions": bear_opinions})
    
    # 결과를 상태에 저장
    state['final_summary'] = final_report.content
    
    # 파일 저장 함수 실행
    save_agent_results(state)

    print("\n=============================================")
    print(" 📄 [최종 투자 리포트 완성] ")
    print("=============================================")
    print(final_report.content)
    
    return {}

In [ ]:
def check_turn(state: AgentState) -> Literal["continue", "end"]:
    current_turn = state.get("turn_count", 0)
    max_turns = state.get("max_turns", 2)
    
    if current_turn >= max_turns:
        return "end"
    else:
        return "continue"

In [ ]:
def route_after_verification(state: AgentState) -> Literal["retry", "next_turn", "end"]:
    """검증 결과에 따라 그래프의 방향을 결정합니다."""
    # 1. 반려 사유가 있다면 무조건 Bull로 돌아가서 턴 다시 시작
    if state.get("rejection_feedback"):
        return "retry"
    
    # 2. 통과했다면 턴 수를 확인하여 분기
    current_turn = state.get("turn_count", 0)
    max_turns = state.get("max_turns", 2)
    
    if current_turn >= max_turns:
        return "end"       # 요약으로 이동
    else:
        return "next_turn" # 다음 턴 진행을 위해 Bull로 이동

In [ ]:
workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node("initialize", initialize)
workflow.add_node("research_node", research_node) # 추가
workflow.add_node("bull_agent", bull_agent)
workflow.add_node("bear_agent", bear_agent)
workflow.add_node("verify_node", verify_node)     # 추가
workflow.add_node("summary_agent", summary_agent)

# 엣지 연결
workflow.add_edge(START, "initialize")
workflow.add_edge("initialize", "research_node")
workflow.add_edge("research_node", "bull_agent")
workflow.add_edge("bull_agent", "bear_agent")
workflow.add_edge("bear_agent", "verify_node") # Bear가 말한 뒤 검증 진행

# 검증이 끝난 후 턴 수를 체크하여 분기
workflow.add_conditional_edges("verify_node", check_turn, {"continue": "bull_agent", "end": "summary_agent"})
workflow.add_edge("summary_agent", END)

# 워크플로우에 라우터 연결
workflow.add_conditional_edges(
    "verify_node",
    route_after_verification,
    {
        "retry": "bull_agent",      # 반려: 다시 작성
        "next_turn": "bull_agent",  # 통과: 다음 턴 시작
        "end": "summary_agent"      # 통과 & 턴 종료: 요약 작성
    }
)

app = workflow.compile()

In [ ]:
app

In [ ]:
# 분석하고자 하는 기업 티커(Ticker) 또는 이름을 입력하세요.
test_topic = "NVIDIA (NVDA)"

# 그래프 실행
# 터미널 창에서 스트리밍으로 로그를 확인하기 위해 stream 사용
inputs = {"topic": test_topic}
for output in app.stream(inputs, stream_mode="updates"):
    # 각 노드가 실행될 때마다 상태 업데이트 내역을 순차적으로 보여줍니다.
    pass